# Sistema de Asistencia Inteligente - Unimarc
## Gestión de Inventario y Operaciones

In [14]:
import os
import time
import re
import json
import numpy as np

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# #1. Cargar variables de entorno
load_dotenv()

base_url = os.getenv("OPENAI_BASE_URL")
api_key = os.getenv("GITHUB_TOKEN")

# #2. Configurar el modelo de lenguaje y la sesión de chat openai
llm = ChatOpenAI(
    base_url=base_url,
    api_key=api_key,
    model="gpt-4o",
    temperature=0.7,
    streaming=True,
    max_tokens=300,
    top_p=1
)

# #2.1 Configurar embeddings para el modelo FAISS
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url=base_url,
    api_key=api_key,
)

# #2.2 Crear la base de conocimiento RAG con vectores FAISS
manual_unimarc_full = """
LOGÍSTICA: El horario de recepción de proveedores en Unimarc es de 08:00 a 13:00 hrs y de 21:00 a 23:00 hrs.
TEMPERATURA: Los productos perecibles deben ingresar con control estricto (máximo 4°C) y mantenerse en las condiciones adecuadas.
MERMAS: Registrar código, cantidad y motivo (vencimiento/rotura) obligatoriamente.
SEGURIDAD: El stock de seguridad mínimo para abarrotes es de 50 unidades por sucursal.
CATEGORÍAS: Pasillos válidos: Abarrotes, Frescos, Lácteos, Bebidas, Congelados, utiles de aseo personal y Limpieza.
REGISTRO: Todo ingreso de stock debe indicar producto, cantidad y unidad de medida.
PROVEEDORES: Los proveedores deben tener autorización previa y estar registrados en el sistema.
DOCUMENTACIÓN: Se requiere factura, guía de despacho y certificado de origen para cada ingreso.
INSPECCIÓN: Verificar estado físico, fechas de vencimiento y códigos de barras.
ALMACENAMIENTO: Productos similares deben estar agrupados en el mismo pasillo por categoría.
"""

# Se divide el documento en fragmentos manejables para mejorar la búsqueda semántica
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = text_splitter.split_text(manual_unimarc_full)

# Se crea la base de datos vectorial FAISS indexada con embeddings
vector_db = FAISS.from_texts(texts=chunks, embedding=embeddings)

# Se configura el retriever para recuperar los chunks más relevantes según la consulta
retriever = vector_db.as_retriever(search_kwargs={"k": 2})


In [15]:
# #3. Crear una estructura para almacenar el historial de conversaciones por sesión
sesion_unimarc = {}

def historial_de_conversacion(sesion_id : str):
    if sesion_id not in sesion_unimarc:
        sesion_unimarc[sesion_id] = InMemoryChatMessageHistory()
    return sesion_unimarc[sesion_id]

# #4. Función para extraer de manera efectiva información de inventario del texto del usuario
def extraer_datos_inventario(texto_usuario):
    prompt_extraccion = (
        "Eres un experto en logística de Unimarc. "
        "Extrae TODOS los datos de inventario del siguiente texto y entrégalos "
        "EXCLUSIVAMENTE en formato JSON puro, sin markdown.\n\n"
        "Instrucciones importantes:\n"
        "- Extrae TODOS los campos disponibles, incluso si faltan algunos pon N/A\n"
        "- Fechas en formato YYYY-MM-DD\n"
        "- Cantidades como números sin unidades\n"
        "- Temperatura con °C o ambiente\n"
        "- Estado: bueno, dañado, vencido\n\n"
        "Formato exacto requerido (sin markdown):\n"
        "{\n"
        "  \"productos\": [\n"
        "    {\n"
        "      \"nombre\": \"nombre del producto\",\n"
        "      \"cantidad\": numero,\n"
        "      \"unidad\": \"kg/unidades/cajas/botellas/litros\",\n"
        "      \"codigo_producto\": \"código SKU\",\n"
        "      \"proveedor\": \"nombre del proveedor\",\n"
        "      \"lote\": \"número de lote\",\n"
        "      \"fecha_vencimiento\": \"YYYY-MM-DD\",\n"
        "      \"temperatura_requerida\": \"temperatura\",\n"
        "      \"categoria\": \"Abarrotes/Frescos/Lacteos/Bebidas/Congelados/Aseo/Limpieza\",\n"
        "      \"estado\": \"bueno/dañado/vencido\"\n"
        "    }\n"
        "  ]\n"
        "}\n\n"
        f"Texto: {texto_usuario}\n"
        "Responde SOLO con el JSON, sin explicaciones, sin markdown:"
    )
    
    try:
        response = llm.invoke([HumanMessage(content=prompt_extraccion)])
        contenido = response.content.strip()
        
        # Limpiar cualquier markdown o asteriscos
        contenido = contenido.replace("```json", "").replace("```", "").strip()
        contenido = contenido.replace("**", "").replace("*", "").strip()
        contenido = contenido.replace("`", "").strip()
        
        # Buscar el JSON
        match = re.search(r'\{[\s\S]*\}', contenido)
        
        if match:
            json_extraido = match.group(0)
            datos_estructurados = json.loads(json_extraido)
            
            # LIMPIAR ASTERISCOS DEL JSON PARSEADO
            for producto in datos_estructurados.get("productos", []):
                for key in producto:
                    if isinstance(producto[key], str):
                        producto[key] = producto[key].replace("**", "").replace("*", "").strip()
            
            return datos_estructurados
        else:
            return {"error": "No se encontró formato JSON en la respuesta.", "raw": contenido}
            
    except json.JSONDecodeError as e:
        return {"error": f"Error al leer el JSON: {str(e)}", "raw": contenido}
    except Exception as e:
        return {"error": f"Ocurrió un error inesperado: {str(e)}"}

# #5. Función para sincronizar el contexto del historial de conversación, resumiendo si es necesario
def sincronizar_contexto_stock(sesion_id: str, max_mensajes=6):
    historial = historial_de_conversacion(sesion_id)

    if len(historial.messages) > max_mensajes:
        mensajes_a_resumir = historial.messages[:-2]
        recientes = historial.messages[-2:]

        conversation_text = ""
        for msj in mensajes_a_resumir:
            role = "Usuario" if msj.type == "human" else "Asistente"
            conversation_text += f"{role}: {msj.content}\n"
        
        prompt_resumen = (
            "Eres un experto en logística de Unimarc. Resume la siguiente conversación. "
            "REGLA DE ORO: No omitas códigos de productos, nombres de artículos ni cantidades de stock. "
            "Resume el resto en máximo 2 líneas de forma técnica.\n\n"
            f"Historial a resumir:\n{conversation_text}"
        )

        summary_response = llm.bind(max_tokens=150).invoke(prompt_resumen)
        summary = summary_response.content
        
        historial.clear()
        historial.add_ai_message(f"[MEMORIA DE STOCK]: {summary}")
        
        for m in recientes:
            if m.type == "human":
                historial.add_user_message(m.content)
            else:
                historial.add_ai_message(m.content)

# #5.1 Función RAG mejorada: Recupera información usando búsqueda vectorial
def recuperar_informacion(query):
    try:
        docs_recuperados = retriever.invoke(query)
        
        if docs_recuperados:
            contenido_rag = "\n".join([doc.page_content for doc in docs_recuperados])
            return contenido_rag
        else:
            return "Usar criterio general de inventario."
    except Exception as e:
        print(f"[ADVERTENCIA]: Error en recuperación RAG: {e}")
        return "Usar criterio general de inventario."

# #6. configuración del prompt con RAG integrado y la función de conversación con memoria
prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres la IA de Gestión de Inventario Unimarc.
RESTRICCIÓN: Solo respondes sobre stock, mermas y logística.

REGLA CRÍTICA: Si el usuario se sale del tema de inventario, RESPONDE EXACTAMENTE: 
"ACCESO DENEGADO: Mi función está restringida exclusivamente a la gestión de inventario de Unimarc."

CONTEXTO RAG: {contexto_rag}

FORMATO OBLIGATORIO para respuestas:
- Sin emojis
- Sin asteriscos o markdown
- Respuestas breves y directas
- Máximo 5 líneas

Usa este formato exacto:

Producto: [nombre]
Cantidad: [cantidad] [unidad]
Codigo: [codigo]
Categoria: [categoria]
Estado: Ingresado correctamente."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

conversation = RunnableWithMessageHistory(
    prompt | llm,
    historial_de_conversacion,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# #7. ejecutar_chat: para manejar la interacción, integrando RAG y sincronización de contexto
def ejecutar_chat(input_text, session_id):
    contexto_recuperado = recuperar_informacion(input_text)
    
    keywords = ["llegó", "anotar", "recibido", "ingresar", "entrada", "stock", "ingresó", "acaba"]
    
    if any(p in input_text.lower() for p in keywords):
        print("[SISTEMA]: Detectado ingreso de inventario...")
        datos = extraer_datos_inventario(input_text)
        
        # Formatear el JSON de forma legible
        if isinstance(datos, dict) and "productos" in datos and datos["productos"]:
            print("\n" + "="*70)
            print("REGISTRO DE INGRESO DE INVENTARIO UNIMARC")
            print("="*70)
            print(f"Hora de registro: {time.strftime('%H:%M:%S')}")
            print(f"Fecha: {time.strftime('%Y-%m-%d')}")
            print("="*70)
            
            for i, producto in enumerate(datos["productos"], 1):
                nombre = producto.get('nombre', 'N/A')
                cantidad = producto.get('cantidad', 'N/A')
                unidad = producto.get('unidad', 'N/A')
                codigo = producto.get('codigo_producto', 'N/A')
                proveedor = producto.get('proveedor', 'N/A')
                lote = producto.get('lote', 'N/A')
                vencimiento = producto.get('fecha_vencimiento', 'N/A')
                temperatura = producto.get('temperatura_requerida', 'Ambiente')
                categoria = producto.get('categoria', 'N/A')
                estado = producto.get('estado', 'Bueno')
                
                print(f"\nPRODUCTO #{i}")
                print(f"  Nombre:              {nombre}")
                print(f"  Cantidad:            {cantidad}")
                print(f"  Unidad:              {unidad}")
                print(f"  Codigo Producto:     {codigo}")
                print(f"  Proveedor:           {proveedor}")
                print(f"  Lote:                {lote}")
                print(f"  Fecha Vencimiento:   {vencimiento}")
                print(f"  Temperatura:         {temperatura}")
                print(f"  Categoria:           {categoria}")
                print(f"  Estado:              {estado}")
            
            print("\n" + "="*70)
            print("PROCESANDO REGISTRO:\n")
            input_final = f"PROCESAR REGISTRO: {json.dumps(datos, indent=2, ensure_ascii=False)}"
        else:
            print("\nError: No se pudieron extraer productos del texto.\n")
            input_final = f"PROCESAR REGISTRO: {datos}"
    else:
        input_final = input_text

    sincronizar_contexto_stock(session_id)
    config = {"configurable": {"session_id": session_id}}
    
    try:
        respuesta_completa = ""
        for chunk in conversation.stream({"input": input_final, "contexto_rag": contexto_recuperado}, config=config):
            # Limpiar asteriscos y markdown de la respuesta
            contenido = chunk.content.replace("**", "").replace("*", "").replace("`", "")
            respuesta_completa += contenido
            print(contenido, end="", flush=True)
        return respuesta_completa
    except Exception as e:
        return f"ERROR: {e}"

# #7.1 Función para mostrar stock completo
def mostrar_stock_completo(session_id):
    """Muestra todo el historial de productos ingresados en la sesión actual"""
    historial = historial_de_conversacion(session_id)
    
    print("\n" + "="*70)
    print("STOCK COMPLETO DE LA SESION")
    print("="*70)
    print(f"Sesion ID: {session_id}")
    print(f"Total de mensajes en memoria: {len(historial.messages)}")
    print("="*70 + "\n")
    
    productos_encontrados = []
    
    # Buscar en el historial
    for mensaje in historial.messages:
        if "PROCESAR REGISTRO" in mensaje.content:
            # Extraer el JSON del mensaje
            try:
                match = re.search(r'\{[\s\S]*\}', mensaje.content, re.DOTALL)
                if match:
                    json_str = match.group(0)
                    datos = json.loads(json_str)
                    if "productos" in datos:
                        productos_encontrados.extend(datos["productos"])
            except:
                pass
    
    if productos_encontrados:
        print(f"Total de productos en stock: {len(productos_encontrados)}\n")
        
        for i, producto in enumerate(productos_encontrados, 1):
            cantidad_total = producto.get('cantidad', 0)
            
            print(f"ITEM #{i}")
            print(f"  Nombre:              {producto.get('nombre', 'N/A')}")
            print(f"  Cantidad:            {cantidad_total} {producto.get('unidad', 'unidades')}")
            print(f"  Codigo Producto:     {producto.get('codigo_producto', 'N/A')}")
            print(f"  Proveedor:           {producto.get('proveedor', 'N/A')}")
            print(f"  Lote:                {producto.get('lote', 'N/A')}")
            print(f"  Fecha Vencimiento:   {producto.get('fecha_vencimiento', 'N/A')}")
            print(f"  Temperatura:         {producto.get('temperatura_requerida', 'Ambiente')}")
            print(f"  Categoria:           {producto.get('categoria', 'N/A')}")
            print(f"  Estado:              {producto.get('estado', 'Bueno')}\n")
        
        # Resumen por categoría
        print("="*70)
        print("RESUMEN POR CATEGORIA:")
        print("="*70)
        
        categorias = {}
        for producto in productos_encontrados:
            cat = producto.get('categoria', 'Sin categoria')
            if cat not in categorias:
                categorias[cat] = []
            categorias[cat].append(producto)
        
        for categoria, productos in categorias.items():
            total_cantidad = sum(p.get('cantidad', 0) for p in productos)
            print(f"\n{categoria}:")
            for producto in productos:
                print(f"  - {producto.get('nombre', 'N/A')}: {producto.get('cantidad', 0)} {producto.get('unidad', 'unidades')}")
            print(f"  Total: {total_cantidad}")
    else:
        print("No hay productos registrados en la sesion actual.\n")
    
    print("="*70 + "\n")

# #8. interfaz blindada para simular la interacción con el sistema de gestión de inventario
print("SISTEMA DE GESTIÓN DE INVENTARIO - UNIMARC")
print("MONITOREO DE SEGURIDAD OPERATIVO ACTIVADO")
print("(Escribe 'stock' para ver el inventario completo)\n")

id_sesion_actual = "admin_test_01"
contador_advertencias = 0 

while True:
    pregunta = input("Usuario: ")
    
    if pregunta.lower() in ["salir", "exit", "quit"]:
        print("SESIÓN FINALIZADA POR EL USUARIO.")
        break
    
    # Comando para ver stock completo
    if pregunta.lower() in ["stock", "inventario", "ver stock", "mostrar stock"]:
        mostrar_stock_completo(id_sesion_actual)
        continue
        
    if pregunta.strip() == "":
        continue
    
    print(f"\n[INPUT]: {pregunta}")
    print(f"[OUTPUT]: ", end="", flush=True)
    
    respuesta_acumulada = ""
    try:
        respuesta_acumulada = ejecutar_chat(pregunta, id_sesion_actual)
        print("\n")
        
        if "ACCESO DENEGADO" in respuesta_acumulada:
            contador_advertencias += 1
            intentos_restantes = 3 - contador_advertencias
            
            if contador_advertencias < 3:
                print(f"ADVERTENCIA DE SEGURIDAD {contador_advertencias}/3: ACTIVIDAD NO AUTORIZADA.")
                print(f"INTENTOS RESTANTES ANTES DEL BLOQUEO DE SESION: {intentos_restantes}\n")
            else:
                print("SISTEMA: SE HA ALCANZADO EL LIMITE DE ADVERTENCIAS PERMITIDAS.")
                print("SISTEMA: CIERRE DE EMERGENCIA EJECUTADO POR PROTOCOLO DE SEGURIDAD.")
                break 
            
    except Exception as e:
        print(f"ERROR CRITICO DEL SISTEMA: {e}")
        break

SISTEMA DE GESTIÓN DE INVENTARIO - UNIMARC
MONITOREO DE SEGURIDAD OPERATIVO ACTIVADO
(Escribe 'stock' para ver el inventario completo)


[INPUT]: Llegó stock: 50 kilos de arroz integral marca Maravilla, código SKU-ARR-001, lote LOTE-2024-156, vencimiento 2027-03-15, proveedor Distribuidora Central, temperatura ambiente, categoría abarrotes, estado bueno
[OUTPUT]: [SISTEMA]: Detectado ingreso de inventario...

REGISTRO DE INGRESO DE INVENTARIO UNIMARC
Hora de registro: 20:22:26
Fecha: 2026-04-25

PRODUCTO #1
  Nombre:              arroz integral marca Maravilla
  Cantidad:            50
  Unidad:              kg
  Codigo Producto:     SKU-ARR-001
  Proveedor:           Distribuidora Central
  Lote:                LOTE-2024-156
  Fecha Vencimiento:   2027-03-15
  Temperatura:         ambiente
  Categoria:           Abarrotes
  Estado:              bueno

PROCESANDO REGISTRO:

Producto: arroz integral marca Maravilla  
Cantidad: 50 kg  
Codigo: SKU-ARR-001  
Categoria: Abarrotes  
Estado: